<a href="https://colab.research.google.com/github/fpellerano/devllm/blob/main/20_4_a_Tools_and_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

To upload a `requirements.txt` file to your Google Colab environment, follow these steps:

1.  **Open the Files pane**: Click the **Folder icon** in the left-hand sidebar.
2.  **Upload your file**:
    *   **Drag and drop** your file directly into the pane.
    *   **OR** click the **Upload to session storage** icon (a file with an upward arrow) to select it from your local machine.

  
>[IMPORTANT] Files uploaded this way are temporary and will be deleted once the session is recycled.

# Tools and Agents with LangGraph

## Agent vs. Chain

Everything we've built so far has been a **chain** — a fixed sequence of steps where
the execution path is determined at design time. You decide what runs and in what order.

An **agent** is different. It uses an LLM as a *reasoning engine* to decide at runtime:
- What information it needs
- Which tool to call to get that information
- Whether it has enough to answer, or needs to take another step

This makes agents capable of handling open-ended tasks that no fixed chain could anticipate.
The tradeoff: agents are less predictable and harder to debug than chains.

This notebook covers:
- What a **tool** is and how to define one
- The **ReAct reasoning loop** that agents follow
- How to give an agent a **system prompt** to shape its behavior
- Handling queries that require **multiple tool calls**
- Extracting **structured output** from agent responses

In [ ]:
!pip install -qU -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 53.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1

In [ ]:
from google.colab import userdata
import os

google_api_key = userdata.get("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = google_api_key

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite-preview",
    temperature=0.0,
    google_api_key=google_api_key,
)

## Data Setup

We use McDonald's Yelp reviews as our knowledge base.
The vector store gives the agent a tool to search through them semantically.

In [ ]:
import pandas as pd
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

!wget -q https://raw.githubusercontent.com/Gaalipour/Opinion-Mining-in-Costumer-Reviews-for-McDonalds-Restaurants/master/McDonalds-Yelp-Sentiment-DFE.csv

NUM_REVIEWS = 20

full_data = pd.read_csv("McDonalds-Yelp-Sentiment-DFE.csv", encoding="ISO-8859-1").head(NUM_REVIEWS)
full_data[["review"]].to_csv("reviews.csv", index=False)

loader = CSVLoader(file_path="./reviews.csv")
docs   = loader.load()

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"trust_remote_code": False},
)

docsearch = Chroma.from_documents(docs, embeddings)
print(f"Vector store ready with {docsearch._collection.count()} documents.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready with 20 documents.


## What Is a Tool?

A **tool** is a function the LLM can call when it needs information or capabilities
it doesn't have internally. Tools give the agent access to the real world:
databases, APIs, calculators, file systems, web searches, etc.

In LangChain, a tool is any Python function decorated with `@tool`.
The LLM never sees the function's code — it only sees:
1. The **function name** (used to call it)
2. The **docstring** (used to decide *when* to call it)
3. The **parameter types** (used to format the call correctly)

> The docstring is the tool's contract with the LLM. Write it clearly.
> Vague docstrings lead to tools being called at the wrong time or not at all.

In [ ]:
from langchain_core.tools import tool


@tool
def search_reviews(query: str) -> str:
    """Search the McDonald's customer review database for reviews related to a topic or keyword.
    Use this when you need to find what customers said about a specific aspect
    (food quality, service, cleanliness, wait times, etc.).
    Returns the 3 most semantically relevant reviews."""
    docs = docsearch.similarity_search_with_score(query, k=3)
    results = []
    for i, (doc, score) in enumerate(docs, 1):
        results.append(f"[Review {i} | similarity: {1-score:.3f}]\n{doc.page_content}")
    return "\n\n".join(results)


@tool
def count_reviews() -> str:
    """Return the total number of customer reviews in the database.
    Use this when the user asks how many reviews exist."""
    count = docsearch._collection.count()
    return f"There are {count} customer reviews in the database."


@tool
def get_review(index: int) -> str:
    """Retrieve a specific review by its index (1-based).
    Use this when you need to read a particular review in full.
    Valid index range: 1 to 20."""
    idx = max(0, min(index - 1, NUM_REVIEWS - 1))
    review = full_data["review"].iloc[idx]
    return f"Review #{index}: {review}"


tools = [search_reviews, count_reviews, get_review]

# The LLM sees this about each tool:
for t in tools:
    print(f"Tool: {t.name}")
    print(f"Description: {t.description[:100]}...")
    print(f"Args: {t.args}\n")

Tool: search_reviews
Description: Search the McDonald's customer review database for reviews related to a topic or keyword.
    Use th...
Args: {'query': {'title': 'Query', 'type': 'string'}}

Tool: count_reviews
Description: Return the total number of customer reviews in the database.
    Use this when the user asks how man...
Args: {}

Tool: get_review
Description: Retrieve a specific review by its index (1-based).
    Use this when you need to read a particular r...
Args: {'index': {'title': 'Index', 'type': 'integer'}}



## The ReAct Reasoning Loop

LangGraph's `create_react_agent` implements the **ReAct** pattern
(Yao et al., 2022): the agent alternates between **reasoning** and **acting**
until it has enough information to give a final answer.

Each iteration of the loop produces two messages:

```
1. AIMessage (with tool_calls)  ← "I need to call search_reviews('food quality')"
2. ToolMessage                  ← the tool's actual output
```

This continues until the agent produces an `AIMessage` with no `tool_calls` —
that's the final answer.

```
HumanMessage: "What do customers say about cleanliness?"
    │
    ▼
AIMessage [tool_call: search_reviews("cleanliness")]
    │
    ▼
ToolMessage [tool result: "Review 1: ... Review 2: ..."]
    │
    ▼
AIMessage [no tool_calls] ← Final answer
```

The agent may loop many times for complex tasks. Each observation informs the next decision.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import SystemMessage

# The system prompt shapes how the agent reasons and communicates.
# Without it the agent behaves generically; with it you get a specialized assistant.
system_prompt = SystemMessage(
    "You are a McDonald's customer insights analyst. "
    "Use the available tools to research customer reviews before drawing any conclusions. "
    "Always base your answers on evidence from the reviews — never speculate. "
    "Be specific: quote or paraphrase what customers actually said."
)

agent = create_react_agent(llm, tools, prompt=system_prompt)
print("Agent ready.")

Agent ready.


/tmp/ipykernel_748/2627948252.py:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


## Running the Agent

### Reading the Trace

The agent's internal reasoning is fully visible in its message history.
The helper below labels each message type so you can follow the loop.

In [ ]:
def print_trace(result):
    """Print the agent's full reasoning trace with labeled message types."""
    for msg in result["messages"]:
        msg_type = type(msg).__name__

        if msg_type == "HumanMessage":
            print(f"[USER]\n{msg.content}\n")

        elif msg_type == "AIMessage":
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                print(f"[AGENT → deciding to use a tool]")
                for tc in msg.tool_calls:
                    args_str = ", ".join(f"{k}={repr(v)}" for k, v in tc["args"].items())
                    print(f"  Call: {tc['name']}({args_str})")
            else:
                print(f"[AGENT → final answer]\n{msg.content}")

        elif msg_type == "ToolMessage":
            preview = msg.content[:200].replace("\n", " ")
            ellipsis = "..." if len(msg.content) > 200 else ""
            print(f"[TOOL: {msg.name}]\n  {preview}{ellipsis}")

        print()

In [ ]:
# Single tool call: a focused question that only needs one search
result = agent.invoke({
    "messages": [{"role": "user", "content": "What do customers say about the cleanliness of McDonald's?"}]
})

print_trace(result)

[USER]
What do customers say about the cleanliness of McDonald's?


[AGENT → deciding to use a tool]
  Call: search_reviews(query='cleanliness')

[TOOL: search_reviews]
  [Review 1 | similarity: -0.425] review: Super slow service, food's terrible like its been sitting and then reheated. Î¾Everything is out, napkins, iced tea  [Review 2 | similarity: -0.463] review: Ter...

[AGENT → final answer]
[{'type': 'text', 'text': 'Based on the available customer reviews, there are negative reports regarding the cleanliness of the McDonald\'s locations mentioned.\n\nSpecifically:\n*   One customer described the restaurant as **"filthy inside."**\n*   Another customer noted that the **"floor was covered with dropped food."**\n*   Additionally, a review mentioned that the restaurant was out of essential items like napkins, which can contribute to a perception of poor maintenance.', 'extras': {'signature': 'EjQKMgG+Pvb7WKKGtKeqy6DOXQbX8JyPi3eTIS/LbhCuNiJATHjfs3Gi/Tuy6sRozWFyyS9o'}}]



### Multi-Step Reasoning

The real power of an agent shows when a task requires **more than one tool call**.
The agent plans, acts, observes the result, and decides what to do next — all autonomously.

The query below requires the agent to use at least two different tools to answer fully.

In [ ]:
# This question requires: count_reviews() + search_reviews() — two separate tool calls
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "How many reviews are in the database? "
            "Then find the top complaints about food quality and summarize them."
        )
    }]
})

print_trace(result)

[USER]
How many reviews are in the database? Then find the top complaints about food quality and summarize them.


[AGENT → deciding to use a tool]
  Call: count_reviews()
  Call: search_reviews(query='food quality complaints')

[TOOL: count_reviews]
  There are 20 customer reviews in the database.

[TOOL: search_reviews]
  [Review 1 | similarity: 0.103] review: As the previous yelpers have already stated, this place is pretty slow. The food is usually cold.. and the service....yeah. you guessed it. Poor.  [Review 2 | si...

[AGENT → final answer]
[{'type': 'text', 'text': 'There are 20 customer reviews in the database.\n\nBased on the reviews, the primary complaints regarding food quality are:\n\n*   **Temperature issues:** Customers have reported that the food is "usually cold."\n*   **Staleness/Reheating:** There are concerns that the food tastes like it has been "sitting and then reheated," leading to a perception that the quality is "terrible."', 'extras': {'signature': 'EjQKMgG+P

In [ ]:
# A question with ambiguity — watch which tool the agent picks and why
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read review number 5 and tell me if the customer had a positive or negative experience."
    }]
})

print_trace(result)

[USER]
Read review number 5 and tell me if the customer had a positive or negative experience.


[AGENT → deciding to use a tool]
  Call: get_review(index=5)

[TOOL: get_review]
  Review #5: Well, it's McDonald's, so you know what the food is. Î¾This review reflects solely on the poor service. Î¾I have been to this location countless times over the years. Î¾They consistently fa...

[AGENT → final answer]
[{'type': 'text', 'text': 'The customer had a **negative experience**.\n\nThe reviewer explicitly stated, "I could not recommend this location any less," and advised others to "take a pass" if they can wait. Their dissatisfaction was primarily focused on the service, describing the staff as "rude" and noting that they "consistently fail on the service end of things."', 'extras': {'signature': 'EjQKMgG+Pvb7a1pTPyejA/PC+4iQPG8OptQWDrVOn2WnMMgFgvSbGPsePmWxaEslE1M3U8fk'}}]



### How the System Prompt Shapes Agent Behavior

The system prompt doesn't just change tone — it changes **what the agent does**.
Compare the same query with two different system prompts.

In [ ]:
# Agent 1: concise analyst
concise_agent = create_react_agent(
    llm, tools,
    prompt=SystemMessage(
        content="You are a concise data analyst. Answer in bullet points only. "
        "Maximum 3 bullets. Use tools to find evidence first."
    )
)

# Agent 2: detailed storyteller
detailed_agent = create_react_agent(
    llm, tools,
    prompt=SystemMessage(
        content="You are a customer experience consultant writing a narrative report. "
        "Use rich, descriptive language. Always quote specific reviews. "
        "Use tools to find evidence first."
    )
)

query = {"messages": [{"role": "user", "content": "What do customers say about the service?"}]}

r1 = concise_agent.invoke(query)

r2 = detailed_agent.invoke(query)


/tmp/ipykernel_748/4032044599.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  concise_agent = create_react_agent(
/tmp/ipykernel_748/4032044599.py:11: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  detailed_agent = create_react_agent(


=== Concise Analyst ===
[{'type': 'text', 'text': '* Customers frequently report slow service and long wait times.\n* Staff are described by some reviewers as lazy, incompetent, or inattentive to specific requests.\n* Negative experiences with service have led some customers to state they will not return to the establishment.', 'extras': {'signature': 'EjQKMgG+Pvb7ketiq/zXUCf0AwIfTgoFJItc9b2a1RTY7dgGT9c0BN1TCoPncLXuyDw2eCis'}}]

=== Detailed Storyteller ===
[{'type': 'text', 'text': 'The customer experience at this location is currently marred by a pervasive sense of frustration and neglect. When analyzing the feedback regarding service, a clear narrative emerges: patrons feel largely ignored, undervalued, and met with a palpable lack of professionalism.\n\nThe sentiment is overwhelmingly negative, with customers describing interactions that range from indifferent to openly hostile. One patron, who has visited the location repeatedly in hopes of a better experience, paints a vivid pict

In [ ]:
import textwrap

def pretty_print_agent(name, result):
    print(f"=== {name} ===")
    # The agent returns a list of messages; the last one is the final answer
    # We extract the 'text' field from the content list
    content = result["messages"][-1].content
    if isinstance(content, list):
        text = content[0].get('text', '')
    else:
        text = content

    print(textwrap.fill(text, width=80))
    print("\n" + "="*40 + "\n")

pretty_print_agent("Concise Analyst", r1)
pretty_print_agent("Detailed Storyteller", r2)

=== Concise Analyst ===
* Customers frequently report slow service and long wait times. * Staff are
described by some reviewers as lazy, incompetent, or inattentive to specific
requests. * Negative experiences with service have led some customers to state
they will not return to the establishment.


=== Detailed Storyteller ===
The customer experience at this location is currently marred by a pervasive
sense of frustration and neglect. When analyzing the feedback regarding service,
a clear narrative emerges: patrons feel largely ignored, undervalued, and met
with a palpable lack of professionalism.  The sentiment is overwhelmingly
negative, with customers describing interactions that range from indifferent to
openly hostile. One patron, who has visited the location repeatedly in hopes of
a better experience, paints a vivid picture of the staff’s demeanor: **"The
order takers tend to be rude, no smiles, and a lot of 'sighs' and 'lip smacking'
when you talk to them."** This lack of basic

## Structured Output from Agent Responses

Agent responses are free-text by default. For downstream processing — dashboards,
APIs, further chains — you often need typed, structured data.

The pattern: run the agent to get findings, then pipe its output through a
`.with_structured_output()` model to extract a validated Pydantic object.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class ReviewInsights(BaseModel):
    topic: str = Field(description="The topic that was researched")
    total_reviews_found: int = Field(description="How many relevant reviews were found")
    positive_points: List[str] = Field(description="Positive things customers mentioned")
    negative_points: List[str] = Field(description="Negative things customers mentioned")
    overall_sentiment: str = Field(description="Overall sentiment: positive, negative, or mixed")

# Step 1: agent researches the topic
research_result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Search for reviews about McDonald's food quality and service. Summarize what you find."
    }]
})
agent_findings = research_result["messages"][-1].content

# Step 2: structured model extracts typed data from the findings
structured_llm = llm.with_structured_output(ReviewInsights)
insights = structured_llm.invoke(
    f"Extract structured insights from the following review analysis:\n\n{agent_findings}"
)

print(f"Topic             : {insights.topic}")
print(f"Reviews found     : {insights.total_reviews_found}")
print(f"Overall sentiment : {insights.overall_sentiment}")
print(f"\nPositive points:")
for p in insights.positive_points:
    print(f"  + {p}")
print(f"\nNegative points:")
for n in insights.negative_points:
    print(f"  - {n}")

Topic             : Food quality and service at a restaurant
Reviews found     : 2
Overall sentiment : negative

Positive points:

Negative points:
  - Food is often cold and tastes reheated
  - Food quality is described as terrible
  - Service is super slow and poor
  - Staff perceived as lazy or incompetent
  - Lack of basic amenities like napkins and iced tea


## Composing Agent + Structured Output as a Chain

We can wrap the two-step pattern (agent → structured extraction) into a reusable
function that behaves like a single call.

In [ ]:
from langchain_core.runnables import RunnableLambda

def research_and_structure(query: str) -> ReviewInsights:
    """Run the agent to research a topic, then extract structured insights."""
    # Step 1: agent does the research
    result = agent.invoke({"messages": [{"role": "user", "content": query}]})
    raw_findings = result["messages"][-1].content

    # Step 2: extract structured data
    return structured_llm.invoke(
        f"Extract structured insights from the following review analysis:\n\n{raw_findings}"
    )

# Run on multiple topics
topics = [
    "Search for reviews about wait times and speed of service. Summarize findings.",
    "Search for reviews about the atmosphere and cleanliness. Summarize findings.",
]

for topic_query in topics:
    print(f"Query: {topic_query[:60]}...")
    insights = research_and_structure(topic_query)
    print(f"  Sentiment : {insights.overall_sentiment}")
    print(f"  Positives : {insights.positive_points[:2]}")
    print(f"  Negatives : {insights.negative_points[:2]}")
    print()

Query: Search for reviews about wait times and speed of service. Su...
  Sentiment : negative
  Positives : ['Some staff members are described as always friendly']
  Negatives : ['Excessive wait times in drive-thru and service windows', 'Frustratingly slow service speed']

Query: Search for reviews about the atmosphere and cleanliness. Sum...
  Sentiment : mixed
  Positives : ['Nice flooring and stacked stone decor', 'Large windows providing a pleasant environment']
  Negatives : ['Dirty floors with dropped food', 'Poorly stocked supplies']



## References

- [Yao et al. (2022) — ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)
- [LangGraph: create_react_agent](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.chat_agent_executor.create_react_agent)
- [LangChain: Tools](https://python.langchain.com/docs/concepts/tools/)
- [LangChain: Agents](https://python.langchain.com/docs/concepts/agents/)
- [LangGraph: Agent Architectures](https://langchain-ai.github.io/langgraph/concepts/agentic_concepts/)